# The Nucleotide Basis of Pyrimethamine Resistance in *pfdhfr*

**A genomics training module — from SNP to phenotype in *Plasmodium falciparum* drug resistance**

### Background

*Plasmodium falciparum* dihydrofolate reductase-thymidylate synthase (**PfDHFR-TS**, gene
`PF3D7_0417200`) is the molecular target of the antimalarial **pyrimethamine**. 

| PDB ID | Genotype | Description |
|---|---|---|
| **3QGT** | Wild type | PfDHFR-TS + NADPH, dUMP, pyrimethamine |
| **1J3J** | Double mutant **C59R + S108N** | Same complex, resistant genotype |

C59R and S108N are two of the four canonical *pfdhfr* resistance codons (the others being N51I
and I164L) that are still actively genotyped in molecular surveillance of sulfadoxine–pyrimethamine
(SP) resistance across sub-Saharan Africa, usually by Sanger sequencing or PCR-RFLP of blood-spot
DNA.

### What this notebook does

Instead of starting from the protein structure, we start from the **gene** and build up:

1. Fetch the real wild-type *pfdhfr* coding sequence (CDS) from NCBI
2. Locate the four canonical resistance codons in that sequence
3. Work out the minimal-edit nucleotide substitution(s) that would produce the C59R and S108N
   amino acid changes, and classify each SNP (transition/transversion, codon position)
4. Build the actual mutant CDS in silico and confirm it translates to the 1J3J genotype
5. Use the wild-type vs mutant CDS to explore the logic behind field PCR-RFLP genotyping assays
6. Design a simple allele-specific PCR primer pair for codon 108
7. Run a toy variant-calling exercise: simulate reads from each allele, align, call the SNP

### Learning objectives

By the end, trainees should be able to:
- Move fluently between amino-acid-level mutation notation (e.g. `S108N`) and nucleotide-level
  SNP notation
- Reason about codon degeneracy and multiple possible mutational paths to the same amino acid change
- Understand the molecular logic behind PCR-RFLP genotyping assays used in the field
- See, end-to-end, how a single nucleotide substitution becomes a drug-resistance genotype call

> **Note on scope:** This module deliberately stays at the DNA/RNA level. A follow-on structural
> module (mapping these substitutions onto the 1J3J vs 3QGT active site) is a natural extension,
> but is out of scope here.


## 1. Setup

We'll rely on Biopython throughout which we'll install using pip.

**Note** the pip command is being called using bash hence the **!** before the command.


In [1]:
!pip install biopython --quiet

from Bio import Entrez, SeqIO
from Bio.Seq import Seq
from Bio.SeqUtils import MeltingTemp as mt
from Bio.Restriction import AllEnzymes, Analysis
import itertools
import pandas as pd

# Entrez requires an email address to identify yourself to NCBI - replace with your own
Entrez.email = "brianbryson39@gmail.com"


## 2. Fetch the wild-type *pfdhfr* reference CDS

`PF3D7_0417200` is a **single-exon** gene on *P. falciparum* chromosome 4.
The encoded protein has 2 functions (Dual domains) where the N-terminal acts as a ***dihydrofolate reductase*** while the C-Terminal funtions as a ***Thymidine Synthase***
The reference sequence (RefSeq) ID for the nucleotide sequence is `NC_004318.2`, spanning positions **748088–749914**. 
The span is exactly 1827 bp which translates to 609 codons (608 amino acid encoding codons plus 1 stop codon), matching the final 608-residue DHFR-TS protein 


This cell fetches it live via NCBI Entrez. If you don't have internet access in the training room
(a real possibility in field settings), the `except` branch lets you paste in a FASTA sequence
you fetched ahead of time and saved locally.

In [2]:
ACCESSION = "NC_004318.2"
START, STOP = 748088, 749914  # 1-based inclusive, plus strand

LOCAL_FASTA_FALLBACK = "pfdhfr_wt_cds.fasta"  # save the fetched record here for offline reuse

try:
    handle = Entrez.efetch(
        db="nuccore",
        id=ACCESSION,
        rettype="fasta",
        retmode="text",
        seq_start=START,
        seq_stop=STOP,
    )
    record = SeqIO.read(handle, "fasta")
    handle.close()
    SeqIO.write(record, LOCAL_FASTA_FALLBACK, "fasta")
    print(f"Fetched {len(record.seq)} bp from NCBI and cached to {LOCAL_FASTA_FALLBACK}")
except Exception as e:
    print(f"Live fetch failed ({e}); trying local cache '{LOCAL_FASTA_FALLBACK}'...")
    record = SeqIO.read(LOCAL_FASTA_FALLBACK, "fasta")

wt_cds = record.seq
print(f"CDS length: {len(wt_cds)} bp -> {len(wt_cds)//3} codons")
assert len(wt_cds) % 3 == 0, "CDS length should be a multiple of 3"


Fetched 1827 bp from NCBI and cached to pfdhfr_wt_cds.fasta
CDS length: 1827 bp -> 609 codons


## 3. Locate the four canonical resistance codons

Codon numbering in the literature (N51I, C59R, S108N, I164L) is 1-based amino-acid position.
For a single-exon CDS with no leading UTR, codon *n* occupies nucleotides
`[(n-1)*3 : (n-1)*3 + 3]` in 0-based Python slicing.

We translate each codon and check it against the expected wild-type residue as a sanity check —
never trust coordinates blindly.


In [3]:
RESISTANCE_CODONS = {
    51: "N",   # Asn  (wild type)
    59: "C",   # Cys  (wild type)
    108: "S",  # Ser  (wild type)
    164: "I",  # Ile  (wild type)
}

def codon_slice(aa_position):
    # 0-based (start, stop) slice for a 1-based amino-acid position
    start = (aa_position - 1) * 3
    return start, start + 3

rows = []
for pos, expected_aa in RESISTANCE_CODONS.items():
    start, stop = codon_slice(pos)
    codon = str(wt_cds[start:stop])
    translated = str(Seq(codon).translate())
    match = "OK" if translated == expected_aa else "MISMATCH - check coordinates!"
    rows.append([pos, codon, translated, expected_aa, match])

df_check = pd.DataFrame(rows, columns=["AA position", "WT codon", "Translated", "Expected", "Check"])
df_check


,AA position,WT codon,Translated,Expected,Check
0,51,AAT,N,N,OK
1,59,TGT,C,C,OK
2,108,AGC,S,S,OK
3,164,ATA,I,I,OK


## 4. From amino acid change to nucleotide change

`C59R` and `S108N` are amino-acid-level shorthand. Because the genetic code is degenerate, more
than one single-nucleotide substitution of the wild-type codon can produce the resistant amino
acid. Rather than assume a specific published SNP from memory, we compute this directly:

For each wild-type codon, enumerate every codon reachable by a **single** nucleotide change,
keep the ones that translate to the resistant amino acid, and classify each as a **transition**
(purine↔purine or pyrimidine↔pyrimidine) or **transversion**.

The minimal-edit-distance path(s) found here are hypotheses to compare against a validated
reference (e.g. a published RFLP protocol or a MalariaGEN variant call) before using them in an
actual diagnostic context — this notebook shows you *how* to reason about it, not a substitute for
that validation step.


In [4]:
BASES = "ACGT"
PURINES = set("AG")

def transition_or_transversion(b1, b2):
    if b1 == b2:
        return None
    same_class = (b1 in PURINES) == (b2 in PURINES)
    return "transition" if same_class else "transversion"

def single_nt_paths_to_aa(wt_codon, target_aa):
    # All single-nucleotide substitutions of wt_codon that translate to target_aa
    paths = []
    for i, wt_base in enumerate(wt_codon):
        for alt_base in BASES:
            if alt_base == wt_base:
                continue
            mut_codon = wt_codon[:i] + alt_base + wt_codon[i+1:]
            if str(Seq(mut_codon).translate()) == target_aa:
                paths.append({
                    "wt_codon": wt_codon,
                    "mut_codon": mut_codon,
                    "codon_position": i + 1,  # 1st/2nd/3rd position within codon
                    "wt_base": wt_base,
                    "mut_base": alt_base,
                    "change_type": transition_or_transversion(wt_base, alt_base),
                })
    return paths

# --- Codon 59: Cys -> Arg ---
start59, stop59 = codon_slice(59)
wt_codon_59 = str(wt_cds[start59:stop59])
paths_59 = single_nt_paths_to_aa(wt_codon_59, "R")

# --- Codon 108: Ser -> Asn ---
start108, stop108 = codon_slice(108)
wt_codon_108 = str(wt_cds[start108:stop108])
paths_108 = single_nt_paths_to_aa(wt_codon_108, "N")

print(f"Codon 59 (WT = {wt_codon_59}, Cys) — single-nt paths to Arg:")
for p in paths_59:
    print(" ", p)

print(f"\nCodon 108 (WT = {wt_codon_108}, Ser) — single-nt paths to Asn:")
for p in paths_108:
    print(" ", p)


Codon 59 (WT = TGT, Cys) — single-nt paths to Arg:
  {'wt_codon': 'TGT', 'mut_codon': 'CGT', 'codon_position': 1, 'wt_base': 'T', 'mut_base': 'C', 'change_type': 'transition'}

Codon 108 (WT = AGC, Ser) — single-nt paths to Asn:
  {'wt_codon': 'AGC', 'mut_codon': 'AAC', 'codon_position': 2, 'wt_base': 'G', 'mut_base': 'A', 'change_type': 'transition'}


If either list above is **empty**, that tells you something important too: it means Cys→Arg
or Ser→Asn at that particular wild-type codon can't be reached by a single nucleotide change, and
the real resistant allele must involve a different route (e.g. a different wild-type codon variant
circulating in the population, or two linked substitutions). This is a genuine feature of codon
degeneracy worth discussing with trainees — the amino-acid-level mutation name doesn't fully
determine the DNA-level event.

For the rest of this notebook we'll take the **first** valid single-nucleotide path found for each
position as our working mutant allele (edit the cell above / pick a different entry from the list
if you want to explore an alternative path).


In [ ]:
chosen_59 = paths_59[0]
chosen_108 = paths_108[0]

print("Chosen path for codon 59:", chosen_59)
print("Chosen path for codon 108:", chosen_108)


## 5. Build the mutant CDS and confirm the 1J3J genotype

Now we splice the chosen single-nucleotide changes into the wild-type CDS and translate the
result, confirming that codons 59 and 108 read out as Arg and Asn — the genotype captured in
1J3J — while everything else stays identical to the wild-type (3QGT-like) sequence.


In [5]:
mutant_cds = list(str(wt_cds))

def apply_mutation(seq_list, aa_position, codon_position, mut_base):
    start, _ = codon_slice(aa_position)
    idx = start + (codon_position - 1)
    seq_list[idx] = mut_base
    return seq_list

mutant_cds = apply_mutation(mutant_cds, 59, chosen_59["codon_position"], chosen_59["mut_base"])
mutant_cds = apply_mutation(mutant_cds, 108, chosen_108["codon_position"], chosen_108["mut_base"])
mutant_cds = Seq("".join(mutant_cds))

wt_protein = wt_cds.translate()
mut_protein = mutant_cds.translate()

print("Residue 59:  WT =", wt_protein[58], " MUT =", mut_protein[58], " (expect C -> R)")
print("Residue 108: WT =", wt_protein[107], " MUT =", mut_protein[107], " (expect S -> N)")

diffs = [(i+1, a, b) for i, (a, b) in enumerate(zip(wt_protein, mut_protein)) if a != b]
print(f"\nTotal amino acid differences between WT and mutant CDS: {len(diffs)}")
print(diffs)


NameError: name 'chosen_59' is not defined

## 6. Summary table across all four canonical resistance codons

Even though 1J3J only carries two of the four canonical *pfdhfr* resistance mutations, they're
almost always discussed together in the surveillance literature (single → double → triple →
quadruple mutant progression: S108N first, then +N51I or +C59R, then +I164L). This table lays out
the genetics side-by-side for reference.


In [ ]:
all_positions = {51: "I", 59: "R", 108: "N", 164: "L"}  # target resistant AA

summary_rows = []
for pos, target_aa in all_positions.items():
    start, stop = codon_slice(pos)
    wt_codon = str(wt_cds[start:stop])
    paths = single_nt_paths_to_aa(wt_codon, target_aa)
    if paths:
        best = paths[0]
        summary_rows.append([
            f"{RESISTANCE_CODONS.get(pos, '?')}{pos}{target_aa}",
            wt_codon, best["mut_codon"], best["codon_position"], best["change_type"],
            "present in 1J3J" if pos in (59, 108) else "not in 1J3J",
        ])
    else:
        summary_rows.append([f"?{pos}{target_aa}", wt_codon, "no single-nt path found",
                              "-", "-", "present in 1J3J" if pos in (59, 108) else "not in 1J3J"])

df_summary = pd.DataFrame(summary_rows, columns=[
    "Mutation", "WT codon", "Candidate mutant codon", "Codon position changed",
    "Transition/transversion", "In 1J3J?"
])
df_summary


## 7. PCR-RFLP logic: restriction sites gained or lost

PCR-RFLP is still a widely used, low-cost field method for genotyping known *pfdhfr* SNPs: amplify
a fragment spanning the codon of interest, digest with a restriction enzyme whose recognition site
overlaps the SNP, and read the resulting fragment sizes on a gel — wild-type and mutant alleles cut
differently.

Here we scan a window around each mutated codon with `Bio.Restriction` to find enzymes whose
recognition site is created or destroyed by the SNP. This shows you the *computational logic*
behind an RFLP assay design — always cross-check any candidate enzyme against a validated,
published protocol (and against enzyme availability/cost) before using it as an actual diagnostic.


In [ ]:
def restriction_diff(wt_seq, mut_seq, window=15):
    # Enzymes that cut one sequence but not the other, within a local window
    wt_sites = Analysis(AllEnzymes, wt_seq).full()
    mut_sites = Analysis(AllEnzymes, mut_seq).full()

    gained, lost = [], []
    for enzyme in AllEnzymes:
        wt_cuts = set(wt_sites.get(enzyme, []))
        mut_cuts = set(mut_sites.get(enzyme, []))
        if wt_cuts and not mut_cuts:
            lost.append(str(enzyme))
        elif mut_cuts and not wt_cuts:
            gained.append(str(enzyme))
    return gained, lost

# Look at a window around codon 108 (WT vs mutant), e.g. +/- 60 bp
from Bio.Seq import Seq as BioSeq

flank = 60
start108, stop108 = codon_slice(108)
lo, hi = max(0, start108 - flank), stop108 + flank

wt_window = BioSeq(str(wt_cds[lo:hi]))
mut_window = BioSeq(str(mutant_cds[lo:hi]))

gained, lost = restriction_diff(wt_window, mut_window)
print("Candidate enzymes whose site is GAINED by the codon 108 mutation:", gained)
print("Candidate enzymes whose site is LOST by the codon 108 mutation:  ", lost)


Repeat the same scan around codon 59 by changing `start108, stop108` above to
`codon_slice(59)` — a good short exercise for trainees to run themselves rather than being handed
the answer.


## 8. Designing an allele-specific PCR primer pair (codon 108)

A simpler, PCR-only alternative to RFLP is **allele-specific PCR (ARMS-PCR)**: design a forward
primer whose 3′ terminal base matches only the wild-type or only the mutant allele. Only the
matching template will amplify efficiently.

This is a minimal illustrative design — real ARMS-PCR assays typically also introduce a deliberate
extra terminal mismatch to sharpen allele specificity, and every primer should be checked for
secondary structure, specificity (BLAST against the genome), and empirically optimized annealing
temperature before use.


In [ ]:
def make_allele_specific_primer(seq, aa_position, codon_position, primer_len=20):
    # 3' end of the primer lands exactly on the SNP position
    start, _ = codon_slice(aa_position)
    snp_index = start + (codon_position - 1)
    primer_start = snp_index - primer_len + 1
    return str(seq[primer_start:snp_index + 1])

wt_primer_108  = make_allele_specific_primer(wt_cds, 108, chosen_108["codon_position"])
mut_primer_108 = make_allele_specific_primer(mutant_cds, 108, chosen_108["codon_position"])

print("WT-specific forward primer:    ", wt_primer_108, f"(Tm ~{mt.Tm_NN(wt_primer_108):.1f} C)")
print("Mutant-specific forward primer:", mut_primer_108, f"(Tm ~{mt.Tm_NN(mut_primer_108):.1f} C)")
print("\n(They differ only at the 3' terminal base - that's the whole trick.)")


## 9. Mini variant-calling bridge

A small conceptual bridge back to the GATK-style pipelines used for real sequencing data. Real
field genotyping of a single SNP is almost always done on a **PCR amplicon** spanning the codon
of interest (exactly the kind of amplicon our primers in Section 8 would generate) rather than by
shotgun-sequencing the whole gene — so we simulate reads drawn from an amplicon window around the
SNP, "align" them naively to the wild-type reference by exact substring matching, and call the SNP
at codon 108 by tallying bases observed at that reference position.

This is obviously not a real aligner — it's meant to make the read-pileup → variant-call logic
concrete before trainees run the same idea at genome scale with `bwa`/`GATK`.


In [ ]:
import random
from collections import Counter

random.seed(42)

snp_index_108 = codon_slice(108)[0] + (chosen_108["codon_position"] - 1)

# Simulate an amplicon-sequencing window around the SNP (e.g. a ~300 bp PCR product),
# rather than shotgun reads across the whole gene - this matches how these SNPs are
# actually genotyped in the field.
AMPLICON_HALF_WIDTH = 150
amp_lo = max(0, snp_index_108 - AMPLICON_HALF_WIDTH)
amp_hi = min(len(wt_cds), snp_index_108 + AMPLICON_HALF_WIDTH)

def simulate_amplicon_reads(seq, region_lo, region_hi, n_reads=40, read_len=100):
    reads = []
    max_start = region_hi - read_len
    for _ in range(n_reads):
        start = random.randint(region_lo, max(region_lo, max_start))
        reads.append((start, str(seq[start:start + read_len])))
    return reads

# half the reads from a "wild-type sample", half from a "mutant sample" -
# mimicking a mixed infection, which is common in the field
wt_reads = simulate_amplicon_reads(wt_cds, amp_lo, amp_hi, n_reads=40)
mut_reads = simulate_amplicon_reads(mutant_cds, amp_lo, amp_hi, n_reads=40)
all_reads = wt_reads + mut_reads

pileup = []
for start, read in all_reads:
    if start <= snp_index_108 < start + len(read):
        base = read[snp_index_108 - start]
        pileup.append(base)

counts = Counter(pileup)
total = sum(counts.values())
print(f"Reads covering the codon-108 SNP position: {total}")
for base, n in counts.items():
    print(f"  {base}: {n}/{total} ({100*n/total:.0f}%)")

print(f"\nExpected WT base: {chosen_108['wt_base']}, expected mutant base: {chosen_108['mut_base']}")
print("A roughly even split here reflects our simulated 50/50 mixed infection - real allele")
print("frequencies from field samples will rarely be this clean, and low-frequency minority")
print("alleles are exactly where read depth and base-quality filtering start to matter.")


## 10. Wrap-up and further directions

- **Real surveillance data**: the same four *pfdhfr* codons analyzed here are tracked at population
  scale by resources like [MalariaGEN Pf7](https://www.malariagen.net/resource/34/) and
  [WWARN](https://www.wwarn.org/) — a natural follow-on exercise is pulling real allele frequency
  data for a region of interest and comparing it to what we derived here from first principles.
- **Structural follow-on**: mapping these same substitutions onto the 1J3J (mutant) vs 3QGT
  (wild-type) active site would show *why* the C59R/S108N genotype reduces pyrimethamine binding —
  a natural Module 2 if this training continues into structural bioinformatics.
- **Validation reminder**: every "candidate" restriction enzyme and primer generated
  computationally in this notebook should be checked against a validated, published assay (or
  validated independently) before any real diagnostic use — this notebook teaches the reasoning,
  not a ready-to-deploy protocol.

### Discussion prompts for trainees
1. Why might the real-world resistant allele at codon 108 not be reachable by a single-nucleotide
   change from *this particular* reference strain's codon — and what would that imply?
2. What are the practical trade-offs between PCR-RFLP, allele-specific PCR, and full amplicon
   sequencing for field genotyping of drug resistance markers?
3. How would you extend the mini variant-calling exercise here to something closer to a real
   Illumina amplicon dataset?
